# FoodPharmer — LoRA fine-tune Llama-3.2-3B-Instruct (Kaggle T4)

**Before running:**
1. Accelerator → GPU T4 x1
2. Add a Kaggle Secret named `HF_TOKEN` (your Hugging Face access token)
3. Upload `sft_dataset.jsonl` as a Kaggle Dataset (see Step 3 cell for the expected path)
4. Accept Meta's license at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

In [ ]:
# ── Cell 1: install dependencies ─────────────────────────────────────────────
!pip install -q \
    "transformers>=4.45.0" \
    "peft>=0.13.0" \
    "accelerate>=0.34.0" \
    scikit-learn \
    matplotlib

In [ ]:
# ── Cell 2: authenticate to Hugging Face via Kaggle secret ───────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
print("Hugging Face login OK")

In [ ]:
# ── Cell 3: imports and config ────────────────────────────────────────────────
import os, json
from pathlib import Path

# Reduces CUDA memory fragmentation — set before any CUDA allocations.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
SFT_PATH   = Path("/kaggle/input/datasets/pranayhedau007/foodpharmer-sft/sft_dataset.jsonl")
OUTPUT_DIR = "/kaggle/working/llama_health_lora"
TORCH_DTYPE = torch.float16

assert torch.cuda.is_available(), "No GPU found — enable GPU T4 in notebook settings."
print(f"GPU count : {torch.cuda.device_count()}")
print(f"GPU 0     : {torch.cuda.get_device_name(0)}")
print(f"VRAM (GPU 0): {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 4: dataset class ─────────────────────────────────────────────────────
# Accepts either a file path OR a pre-loaded list of records (for train/test splits).
# Prompt template and formatting logic are unchanged from the local script.
class SFTDataset(Dataset):
    def __init__(self, tokenizer, data_path=None, records=None, max_length: int = 640):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.samples: list[str] = []

        source = records if records is not None else [
            json.loads(line) for line in open(data_path)
        ]

        for item in source:
            messages = item["messages"]
            try:
                text = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=False,
                )
            except Exception:
                user      = messages[0]["content"]
                assistant = messages[1]["content"]
                text = (
                    f"<|begin_of_text|><|start_header_id|>user"
                    f"<|end_header_id|>\n\n{user}<|eot_id|>"
                    f"<|start_header_id|>assistant<|end_header_id|>\n\n"
                    f"{assistant}<|eot_id|>"
                )
            self.samples.append(text)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        enc = self.tokenizer(
            self.samples[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids      = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        labels         = input_ids.clone()
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
# ── Cell 5: load tokenizer and model ─────────────────────────────────────────
if not SFT_PATH.exists():
    raise FileNotFoundError(
        f"{SFT_PATH} not found.\n"
        f"Upload sft_dataset.jsonl as a Kaggle Dataset with slug '{DATASET_SLUG}'."
    )

print(f"Loading tokenizer from {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading base model in fp16 ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
    device_map="auto",       # maps all layers to the single T4
    low_cpu_mem_usage=True,
)

# Required before gradient_checkpointing to avoid grad-fn warnings with PEFT
model.enable_input_require_grads()

print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell 6: apply LoRA (same config as local script) ─────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Cell 7: stratified train / test split ────────────────────────────────────
from collections import Counter
from sklearn.model_selection import train_test_split

if not SFT_PATH.exists():
    raise FileNotFoundError(
        f"{SFT_PATH} not found.\n"
        "Update SFT_PATH in Cell 3 to the path shown by os.walk('/kaggle/input')."
    )

all_records = [json.loads(l) for l in SFT_PATH.read_text().splitlines()]

# Extract risk_level from each assistant response for stratification.
# Every record in this dataset has a valid, parseable risk_level.
risk_labels = [json.loads(r["messages"][1]["content"])["risk_level"] for r in all_records]

# Stratified 85 / 15 split — preserves LOW/MEDIUM/HIGH ratio in both sets.
train_records, test_records, train_labels, test_labels = train_test_split(
    all_records, risk_labels,
    test_size=0.15, random_state=42, stratify=risk_labels,
)

print(f"Total : {len(all_records)}  →  Train : {len(train_records)}  |  Test : {len(test_records)}")
print(f"Train distribution : {dict(sorted(Counter(train_labels).items()))}")
print(f"Test  distribution : {dict(sorted(Counter(test_labels).items()))}")

train_dataset = SFTDataset(tokenizer, records=train_records)
test_dataset  = SFTDataset(tokenizer, records=test_records)

In [ ]:
# ── Cell 8: training args and trainer ────────────────────────────────────────
# T4 has 14.56 GiB usable (not full 16 GiB — OS overhead).
# Memory budget at batch=2, seq=640, fp16 + gradient_checkpointing:
#   Model weights fp16   : ~6.2 GB
#   Activations (batch=2): ~1.5 GB   (halved vs batch=4)
#   LoRA optimizer states: ~108 MB
#   Total                : ~8 GB  — fits comfortably
#
# Effective batch = per_device_train_batch_size * gradient_accumulation_steps
#                 = 2 * 8 = 16  (same as before)
#
# Steps math: ~307 train samples / batch 2 = 154 data batches/epoch
#             / grad_accum 8 = ~20 optimizer steps/epoch × 3 epochs = ~60 steps
# eval_steps=10 → ~6 eval points across the run
#
# eval_strategy is the current kwarg (transformers ≥ 4.41).
# Older versions need evaluation_strategy instead.

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=5,
    eval_strategy="steps",       # older transformers: evaluation_strategy="steps"
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    do_eval=True,
    report_to=[],
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nBest checkpoint kept at: {OUTPUT_DIR}")
print(f"VRAM after training: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Cell A: generate predictions on the held-out test set ────────────────────
# Runs the fine-tuned model on each test input, parses the JSON output,
# and extracts risk_level.  Parse failures are counted, not crashed on.

import re

model.eval()

true_labels  = []
pred_labels  = []
parse_failed = []   # (index, raw_output) for unparseable responses

LABEL_SET = {"LOW", "MEDIUM", "HIGH"}

print(f"Running inference on {len(test_records)} test samples ...")

for i, record in enumerate(test_records):
    user_msg   = record["messages"][0]["content"]
    true_label = json.loads(record["messages"][1]["content"])["risk_level"]
    true_labels.append(true_label)

    # Format prompt for generation (add_generation_prompt=True tells the model to start its reply)
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_msg}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=640
    ).to("cuda:0")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (strip the echoed prompt)
    gen_ids  = output_ids[0][inputs["input_ids"].shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    # Try to parse JSON and extract risk_level
    pred = None
    try:
        # Model may wrap JSON in markdown fences — strip them
        cleaned = re.sub(r"^```(?:json)?\s*", "", gen_text, flags=re.IGNORECASE)
        cleaned = re.sub(r"\s*```$", "", cleaned).strip()
        obj  = json.loads(cleaned)
        pred = obj.get("risk_level", "").upper()
        if pred not in LABEL_SET:
            pred = None
    except Exception:
        pass

    if pred is None:
        parse_failed.append({"index": i, "true": true_label, "raw_output": gen_text[:300]})

    pred_labels.append(pred if pred is not None else "PARSE_ERROR")

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(test_records)} done")

print(f"\nDone. Parse failures: {len(parse_failed)} / {len(test_records)}")
if parse_failed:
    for pf in parse_failed:
        print(f"  sample {pf['index']} (true={pf['true']}): {pf['raw_output'][:120]}")

In [ ]:
# ── Cell B: classification metrics ───────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Exclude parse failures from metric computation (they're already counted separately)
valid_mask  = [p != "PARSE_ERROR" for p in pred_labels]
y_true      = [t for t, v in zip(true_labels,  valid_mask) if v]
y_pred      = [p for p, v in zip(pred_labels, valid_mask) if v]
n_valid     = len(y_true)
n_parsefail = len(parse_failed)

CLASSES = ["HIGH", "LOW", "MEDIUM"]   # alphabetical for sklearn consistency

accuracy = accuracy_score(y_true, y_pred)
report   = classification_report(y_true, y_pred, labels=CLASSES, output_dict=True)
macro_f1 = report["macro avg"]["f1-score"]
cm       = confusion_matrix(y_true, y_pred, labels=CLASSES)

print(f"Evaluated on {n_valid} parseable samples  ({n_parsefail} parse failures excluded)\n")
print(f"Accuracy : {accuracy:.4f}")
print(f"Macro-F1 : {macro_f1:.4f}   ← headline metric (accounts for class imbalance)\n")
print(classification_report(y_true, y_pred, labels=CLASSES))

# Confusion matrix plot
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — risk_level (test set)")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150)
plt.show()
print("Saved confusion_matrix.png")

In [ ]:
# ── Cell C: error analysis — every misclassified test example ─────────────────
misclassified = []
for i, (true, pred, record) in enumerate(zip(true_labels, pred_labels, test_records)):
    if true != pred:
        user_content = record["messages"][0]["content"]
        # Extract ingredient list line from the prompt
        lines = user_content.splitlines()
        ingr_line = next((l for l in lines if l.strip() and "Ingredient list" not in l
                          and "Your response" not in l and "schema" not in l
                          and "health_score" not in l and "You are" not in l), "")
        misclassified.append({
            "test_index"  : i,
            "true_label"  : true,
            "pred_label"  : pred,
            "ingredients" : ingr_line.strip()[:120],
        })

print(f"Misclassified: {len(misclassified)} / {len(test_records)}\n")
if misclassified:
    header = f"{'#':>3}  {'True':<8}  {'Pred':<12}  Ingredients (truncated)"
    print(header)
    print("-" * len(header))
    for row in misclassified:
        print(f"{row['test_index']:>3}  {row['true_label']:<8}  {row['pred_label']:<12}  {row['ingredients']}")

In [ ]:
# ── Cell D: save all artifacts to /kaggle/working/ ────────────────────────────
import csv

working = Path("/kaggle/working")

# 1. Train/eval loss history from Trainer log
log_history = trainer.state.log_history
loss_rows = []
for entry in log_history:
    row = {
        "step"      : entry.get("step"),
        "train_loss": entry.get("loss"),
        "eval_loss" : entry.get("eval_loss"),
        "epoch"     : entry.get("epoch"),
    }
    if row["train_loss"] is not None or row["eval_loss"] is not None:
        loss_rows.append(row)

loss_csv = working / "loss_history.csv"
with open(loss_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["step","epoch","train_loss","eval_loss"])
    writer.writeheader()
    writer.writerows(loss_rows)
print(f"Saved {loss_csv}  ({len(loss_rows)} rows)")

# 2. Metrics summary JSON
metrics_summary = {
    "test_samples_total"   : len(test_records),
    "parse_failures"       : n_parsefail,
    "evaluated_samples"    : n_valid,
    "accuracy"             : round(accuracy, 4),
    "macro_f1"             : round(macro_f1, 4),
    "per_class"            : {
        cls: {
            "precision": round(report[cls]["precision"], 4),
            "recall"   : round(report[cls]["recall"],    4),
            "f1"       : round(report[cls]["f1-score"],  4),
            "support"  : report[cls]["support"],
        }
        for cls in CLASSES if cls in report
    },
    "confusion_matrix": {
        "labels": CLASSES,
        "counts": cm.tolist(),
    },
    "misclassified_count": len(misclassified),
}

metrics_json = working / "metrics_summary.json"
metrics_json.write_text(json.dumps(metrics_summary, indent=2))
print(f"Saved {metrics_json}")

# 3. Misclassified examples CSV
misclass_csv = working / "misclassified_examples.csv"
with open(misclass_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["test_index","true_label","pred_label","ingredients"])
    writer.writeheader()
    writer.writerows(misclassified)
print(f"Saved {misclass_csv}  ({len(misclassified)} rows)")

# 4. Confusion matrix raw counts JSON (already embedded in metrics_summary, but standalone too)
cm_json = working / "confusion_matrix_counts.json"
cm_json.write_text(json.dumps({"labels": CLASSES, "counts": cm.tolist()}, indent=2))
print(f"Saved {cm_json}")

# confusion_matrix.png was saved in Cell B
print(f"Saved /kaggle/working/confusion_matrix.png  (plot)")

In [ ]:
# ── Cell E: final summary ─────────────────────────────────────────────────────
# Extract final train loss and best eval loss from log history
train_losses = [e["loss"]      for e in log_history if "loss"      in e]
eval_losses  = [e["eval_loss"] for e in log_history if "eval_loss" in e]

final_train_loss = train_losses[-1] if train_losses else float("nan")
best_eval_loss   = min(eval_losses)  if eval_losses  else float("nan")

print("=" * 55)
print("  TRAINING SUMMARY")
print("=" * 55)
print(f"  Train samples       : {len(train_records)}")
print(f"  Test  samples       : {len(test_records)}")
print(f"  Final train loss    : {final_train_loss:.4f}")
print(f"  Best  eval  loss    : {best_eval_loss:.4f}")
print("-" * 55)
print("  TEST SET METRICS  (risk_level classification)")
print(f"  Accuracy            : {accuracy:.4f}")
print(f"  Macro-F1            : {macro_f1:.4f}  ← headline")
for cls in CLASSES:
    if cls in report:
        r = report[cls]
        print(f"    {cls:<8}  P={r['precision']:.2f}  R={r['recall']:.2f}  "
              f"F1={r['f1-score']:.2f}  (n={int(r['support'])})")
print("-" * 55)
print(f"  Parse failures      : {n_parsefail} / {len(test_records)}")
print(f"  Misclassified       : {len(misclassified)} / {len(test_records)}")
print("=" * 55)
print("\nArtifacts in /kaggle/working/")
print("  loss_history.csv          — step-by-step train/eval loss")
print("  metrics_summary.json      — all numbers above")
print("  confusion_matrix.png      — visual confusion matrix")
print("  confusion_matrix_counts.json")
print("  misclassified_examples.csv")

In [ ]:
# ── Cell 9: verify saved files ────────────────────────────────────────────────
import os

saved = list(Path(OUTPUT_DIR).rglob("*"))
print(f"Files in {OUTPUT_DIR}:")
for p in sorted(saved):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f"  {p.relative_to(OUTPUT_DIR)}  ({size_mb:.1f} MB)")

# The adapter_model.safetensors (or adapter_model.bin) is what you download
# and load locally with PeftModel.from_pretrained().